In [585]:
# Author: Niko Bleidistel
# last change: 2026-08-04

# Package Import

In [ ]:
from pathlib import Path 
from os import makedirs
import sys
import importlib
import re

import pandas as pd
import numpy as np

import mph
import matplotlib as mpl
import matplotlib.pyplot as plt

In [587]:
PYTHON_HELPER_FOLDER = Path(r"py-helpers")

# Add the path to the custom packages to sys.path so that they can be imported
sys.path.append(str(PYTHON_HELPER_FOLDER.resolve()))

# import custom packages
import comsol_data_export as cde
import comsol_data_plotting as cdp
import plot_functions as pfs
import time_logging as tl

# reload custom packages (for each execution) to reflect recent changes
_ = importlib.reload(cde)
_ = importlib.reload(cdp)
_ = importlib.reload(pfs)
_ = importlib.reload(tl)

# PATHS

In [588]:
# INPUT_FOLDER = Path(r"R:\Bleidistel_Niko\COMSOL\COMSOL Files\06_mfco_assymmetry")
# OUTPUT_FOLDER = Path(r"R:\Bleidistel_Niko\COMSOL\COMSOL Script Outputs\000_New_Output")

MAIN_FOLDER = Path(r"R:\Bleidistel_Niko\COMSOL\COMSOL Files\10_bachelor_thesis_models_use_terminals")
INPUT_FOLDER = MAIN_FOLDER / "Solved model versions"
OUTPUT_FOLDER = MAIN_FOLDER / "Test Output"

makedirs(OUTPUT_FOLDER, exist_ok=True)  # create output folder if it doesn't exist

# INITIALIZE

In [589]:
# initialize time logging
_ = tl.initialize_time_log(OUTPUT_FOLDER / 'time_log.csv')

# initialize COMSOL client (server)
client = mph.start()

# SIMULATE

## just solving model

In [590]:
# just solve the models from a input folder and save the solved models to an output folder in the same directory 
if False:
    input_folder = MAIN_FOLDER
    output_folder = INPUT_FOLDER
    makedirs(output_folder, exist_ok=True)  # create output folder if it doesn't exist

    mph_files = list(input_folder.glob("*.mph"))

    print("List of model files to be processed:")
    for modelfile in mph_files:
            print(modelfile.stem)

    print("\nCurrently processing...\n")
    for modelfile in mph_files:
        print(modelfile.stem)
        try:
            cde.simulate_model(
                    # path settings
                    filename = modelfile.stem,
                    input_folder = modelfile.parent,
                    output_folder = output_folder,
            
                    # simulation settings
                    client = client,
                    export_params = [],
                    export_descriptions = [],
            
                    # boolean flags    
                    export_parameters_to_csv = True,
                    extend_export_from_params_in_csv= False,
                    show_model_info = False,
                    solve_model = True,
                    save_solved_model = True,
                    export_all_solution_data = False,
                    save_small_model_version = False,
                    new_log_file = True,
                )
        except Exception as e:
            with open(output_folder / 'error_log.txt', 'a') as f:
                f.write(f"Error occurred while processing {modelfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {modelfile.name}")
        print(f"Finished.\n")

## Constants

In [591]:
EXPORT_DICT = {
    "mf.normB": "Magnetic flux density, norm [T]",
    "mf.Bx": "Magnetic flux density, x-component [T]", 
    "mf.By": "Magnetic flux density, y-component [T]", 
    "mf.Bz": "Magnetic flux density, z-component [T]",
    "T": "Temperature [K]",
}
EXPORT_PARAMS = list(EXPORT_DICT.keys())
EXPORT_DESCRIPTION = list(EXPORT_DICT.values())

CONDUCTOR_EXPORT_DICT = {
    "V": "Electric potential [V]",
    "ec.normJ": "Current density, norm [A/m^2]",
    "ec.Jx": "Current density, x-component [A/m^2]",
    "ec.Jy": "Current density, y-component [A/m^2]",
    "ec.Jz": "Current density, z-component [A/m^2]",
}
CONDUCTOR_EXPORT_PARAMS = list(CONDUCTOR_EXPORT_DICT.keys())
CONDUCTOR_EXPORT_DESCRIPTION = list(CONDUCTOR_EXPORT_DICT.values())

## Simulate

In [ ]:
if True:
    input_folder = INPUT_FOLDER
    output_folder = OUTPUT_FOLDER

    mph_files = list(input_folder.glob("*.mph"))

    print("List of model files to be processed:")
    for modelfile in mph_files:
            print(modelfile.stem)

    print("\nCurrently processing...\n")
    for modelfile in mph_files:
        print(modelfile.stem)

        model_output_folder = output_folder / modelfile.stem
        makedirs(model_output_folder, exist_ok=True)  # create output folder if it doesn't exist

        try:
            cde.simulate_model(
                    # path settings
                    filename = modelfile.stem,
                    input_folder = modelfile.parent,
                    output_folder = model_output_folder,

                    # simulation settings
                    client = client,
                    export_params = EXPORT_PARAMS.copy(),
                    export_descriptions = EXPORT_DESCRIPTION.copy(),
                    conductor_export_params = CONDUCTOR_EXPORT_PARAMS.copy(),
                    conductor_export_descriptions = CONDUCTOR_EXPORT_DESCRIPTION.copy(),

                    # COMSOL internal interpolation
                    Depth_point1 = (0.0, 0.0, 0.0),
                    Depth_point2 = (0.0, 0.0, "-1*epilayer_height"),

                    Homogeneity_point1 = (0.0, "+0.5*substrate_length", 0.0),
                    Homogeneity_point2 = (0.0, "-0.5*substrate_length", 0.0),
                    Homogeneity_distances = "range(0,(-1*epilayer_height-0)/9,-1*epilayer_height)",
                    Homogeneity_orth_vector = [0, 0, 1],

                    Longitudinal_point1 = ("+0.5*substrate_length", 0.0, 0.0),
                    Longitudinal_point2 = ("-0.5*substrate_length", 0.0, 0.0),
                    Longitudinal_distances = "range(0,(-1*epilayer_height-0)/9,-1*epilayer_height)",
                    Longitudinal_orth_vector = [0, 0, 1],

                    # boolean flags    
                    export_parameters_to_csv = True,
                    evaluate_parameter_expressions = True,
                    extend_export_from_params_in_csv= False,
                    show_model_info = False,
                    solve_model = False, # already solved models are used here
                    save_solved_model = True,
                    export_all_solution_data = False, # file size is pretty large
                    export_line_solution_data = True,
                    export_plane_solution_data = True,
                    save_small_model_version = False,
                    new_log_file = True,
            )
    
        except Exception as e:
            with open(model_output_folder / 'errormessage.txt', 'a') as f:
                f.write(f"Error occurred while processing {modelfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {modelfile.name}")
        print(f"Finished.\n")

## TODO: 
- standard plots erstellen
- sweep function auf stand von simulations function bringen (helper functions einbauen ?)
    - number turns 15, 25, 35
    - 3 grad schritte bei angle error (0 bis 90) 
- modelle updaten 
    - Terminals einrichten 
    - Ströme = 10mA
    - conductor_all_height = 1[um]
    - conductor_grid_length 60[um]
- spezielle eigenschaften im workflow berücksichtigen (insulator und sweeps)

# PLOT

## Constants

In [593]:
TRANSLATE_PLOTLABELS = {
    "x":                r"$x$-axis [m]",
    "y":                r"$y$-axis [m]",
    "z":                r"$z$-axis [m]",
    "mf.normB (T)":     r"Magnetic flux density, norm $|\vec{B}|$ [T]",
    "mf.Bx (T)":        r"Magnetic flux density, x-component $B_x$ [T]", 
    "mf.By (T)":        r"Magnetic flux density, y-component $B_y$ [T]", 
    "mf.Bz (T)":        r"Magnetic flux density, z-component $B_z$ [T]",
    "T (K)":            r"Temperature $T$ [K]",
    "V (V)":            r"Electric potential $V$ [V]",
    "ec.normJ (A/m^2)": r"Current density, norm $|\vec{J}|$ [A/m^2]",
    "ec.Jx (A/m^2)":    r"Current density, x-component $J_x$ [A/m^2]",
    "ec.Jy (A/m^2)":    r"Current density, y-component $J_y$ [A/m^2]",
    "ec.Jz (A/m^2)":    r"Current density, z-component $J_z$ [A/m^2]"
}

In [594]:
X_AXIS_PARAMS = ["x", "y", "z"]
Y_AXIS_PARAMS = [key for key in TRANSLATE_PLOTLABELS.keys() if key not in X_AXIS_PARAMS]

In [595]:
if False:
    input_folder = OUTPUT_FOLDER
    output_folder = OUTPUT_FOLDER

    model_output_dirs = [f for f in output_folder.iterdir() if f.is_dir()]

    model_counter = 1
    print("List of Data to be processed:")
    for model_output_dir in model_output_dirs:
        print("="*60)
        print(f"{model_counter}: {model_output_dir.stem}")
        file_counter = 1

        model_data_dir = model_output_dir / "Data Export"
        txt_files = list(model_data_dir.glob("*.txt"))
        print("-"*60)
        for txt_file in txt_files:
            print(f"{model_counter}.{file_counter}: {txt_file.stem}")
            file_counter += 1
        print("="*60)
        model_counter += 1
    

In [596]:
if True:
    input_folder = OUTPUT_FOLDER
    output_folder = OUTPUT_FOLDER

    # get all subfolders in the input folder (each subfolder corresponds to a model)
    model_input_dirs = [f for f in input_folder.iterdir() if f.is_dir()]

    print("List of Data to be processed:")
    model_counter = 1
    for model_intput_dir in model_input_dirs:
        print("="*60)
        print(f"{model_counter}: {model_intput_dir.stem}")
        tl.log_message(f"Started importing of solution data in {model_intput_dir.stem}")

        # define output folder for plots
        plot_folder = model_intput_dir / "Plots"
        # from each model output folder, get the "Data Export" subfolder
        model_data_dir = model_intput_dir / "Data Export"

        # get a list of all csv files in the "Data Export" subfolder
        df_terminal = pd.DataFrame()
        df_parameters = pd.DataFrame()

        csv_files = list(model_data_dir.glob("*.csv"))
        for csv_file in csv_files:
            if re.match(r".*-terminals$", csv_file.stem):
                df_terminal = pd.read_csv(csv_file)
                tl.log_message(f"Imported terminal data")
            elif re.match(r".*-parameters$", csv_file.stem):
                df_parameters = pd.read_csv(csv_file)
                tl.log_message(f"Imported parameter data")

        # get a list of all txt files in the "Data Export" subfolder
        txt_files = list(model_data_dir.glob("*.txt"))
        print("-"*60)
        file_counter = 1
        for txt_file in txt_files:
            print(f"{model_counter}.{file_counter}: {txt_file.stem}")

            header_data, df = cdp.read_comsol_export(str(txt_file))
            modelstem = str(header_data.get('Model')).replace(".mph", "")
            modeltitle = re.match(r"(.*)-(.*)-.*", modelstem).group(2)

            if re.match(r".*-depth_exported_data$", txt_file.stem):
                # import data from txt file
                df.drop(labels=["x", "y"], axis=1, inplace=True)

                depth_folder = plot_folder / "zaxis"
                makedirs(depth_folder, exist_ok=True)  # create output folder if it doesn't exist

                xparam = "z"
                for yparam in Y_AXIS_PARAMS:
                    if yparam in df.columns:
                        tl.log_message(f"Started plotting of {yparam} vs {xparam}")
                        fig, ax = cdp.standard_plot(
                                        output_folder = str(depth_folder),
                                        x_param = xparam,
                                        y_param = yparam,
                                        header_data = header_data,
                                        df_curves = df,
                                        df_param = df_parameters,
                                        title = modeltitle,
                                        add_title_info = False,
                                        translation_dict = TRANSLATE_PLOTLABELS,
                                        color = 'midnightblue',
                                        save_plot = True,
                                        )
                        tl.log_message(f"Finished plotting of {yparam} vs {xparam}")



            if re.match(r".*-homogeneity_exported_data$", txt_file.stem):
                            # import data from txt file
                            df.drop(labels=["x"], axis=1, inplace=True)

                            # create a copy of the original DataFrame and round the "z" values to 6 significant digits to avoid floating point precision issues
                            df_copy = df.copy()  
                            df_copy["z"] = cde.round_to_6_sig_digits(df["z"])
                            unique_z_values = df_copy["z"].dropna().unique()
            
                            cmap = mpl.colormaps['viridis'] # type: ignore
            
                            xparam = "y"
                            homogeneity_folder = plot_folder / "yaxis"
                            makedirs(homogeneity_folder, exist_ok=True)  # create output folder if it doesn't exist

                            for yparam in Y_AXIS_PARAMS:
                                if yparam in df.columns:
                                    tl.log_message(f"Started plotting of {yparam} vs {xparam}")
            
                                    fig, ax = None, None  
                                    for loop_counter, z_value in enumerate(unique_z_values):
                                        df_subset = df_copy[df_copy["z"] == z_value]
                                        color = cmap(loop_counter / len(unique_z_values))  # get color from colormap based on index
                    
                                        fig, ax = cdp.standard_plot(
                                                        output_folder = None,
                                                        x_param = xparam,
                                                        y_param = yparam,
                                                        header_data = header_data,
                                                        df_curves = df_subset,
                                                        df_param = df_parameters,
                                                        sweep_params = ["z"],
                                                        title = modeltitle,
                                                        add_title_info = False,
                                                        translation_dict = TRANSLATE_PLOTLABELS,
                                                        fig = fig, 
                                                        ax = ax,
                                                        color=color,
                                                        save_plot = False,
                                                        )
                                        ax.legend(
                                            loc="upper center", 
                                            bbox_to_anchor=(0.5, -0.15), 
                                            ncol=2
                                            )
                                        xlength = df_parameters[df_parameters["name"] == "conductor_all_length"]["evaluated_value"].item()
                                        ax.set_xlim(left=-0.5*xlength, right=0.5*xlength)
            
                                        
                                        modelname = str(header_data.get('Model')).replace(".mph", "")
                                        output_path = homogeneity_folder / f"{modelname}_{xparam}_vs_{yparam}.png"
                                        fig.savefig(str(output_path), dpi=300)
                                    tl.log_message(f"Finished plotting of {yparam} vs {xparam}")



            if re.match(r".*-longitudinal_exported_data$", txt_file.stem):
                # import data from txt file
                df.drop(labels=["y"], axis=1, inplace=True)

                # create a copy of the original DataFrame and round the "z" values to 6 significant digits to avoid floating point precision issues
                df_copy = df.copy()
                df_copy["z"] = cde.round_to_6_sig_digits(df["z"])
                unique_z_values = df_copy["z"].dropna().unique()

                cmap = mpl.colormaps['viridis'] # type: ignore

                xparam = "x"
                longitudinal_folder = plot_folder / "xaxis"
                makedirs(longitudinal_folder, exist_ok=True)  # create output folder if it doesn't exist

                for yparam in Y_AXIS_PARAMS:
                    if yparam in df.columns:
                        tl.log_message(f"Started plotting of {yparam} vs {xparam}")

                        fig, ax = None, None  
                        for loop_counter, z_value in enumerate(unique_z_values):
                            df_subset = df_copy[df_copy["z"] == z_value]
                            color = cmap(loop_counter / len(unique_z_values))  # get color from colormap based on index
                            
                            fig, ax = cdp.standard_plot(
                                            output_folder = None,
                                            x_param = xparam,
                                            y_param = yparam,
                                            header_data = header_data,
                                            df_curves = df_subset,
                                            df_param = df_parameters,
                                            sweep_params = ["z"],
                                            title = modeltitle,
                                            add_title_info = False,
                                            translation_dict = TRANSLATE_PLOTLABELS,
                                            fig = fig, 
                                            ax = ax,
                                            color=color,
                                            save_plot = False,
                                            )
                            ax.legend(
                                loc="upper center", 
                                bbox_to_anchor=(0.5, -0.15), 
                                ncol=2
                                )
                            xlength = df_parameters[df_parameters["name"] == "conductor_all_length"]["evaluated_value"].item()
                            ax.set_xlim(left=-0.5*xlength, right=0.5*xlength)

                            
                            modelname = str(header_data.get('Model')).replace(".mph", "")
                            output_path = longitudinal_folder / f"{modelname}_{xparam}_vs_{yparam}.png"
                            fig.savefig(str(output_path), dpi=300)
                        tl.log_message(f"Finished plotting of {yparam} vs {xparam}")

                        
            plt.close('all')


            file_counter += 1
        print("="*60)
        model_counter += 1

List of Data to be processed:
1: 01_03_b-Rectangular spiral combined with grid-solved
------------------------------------------------------------
1.1: 01_03_b-Rectangular spiral combined with grid-solved-conductor_exported_data


KeyboardInterrupt: 

## OLD: Interpolate

In [ ]:
if False:
    def import_solution_data(
            # path settings
            filename: str, # f"{modelname}_exported_data.txt"
            input_folder: Path,
            ):

        txt_path = input_folder / filename
        tl.log_message(f"Started importing of solution data in {txt_path.name}")

        
        # import data from txt file
        header_data, df = cdp.read_comsol_export(str(txt_path))
        constants_df = cde.find_constant_columns(df)

        # get original modelname
        modelstem = str(header_data.get('Model')).replace(".mph", "")
        tl.log_message(f"Imported data shows original modelname {modelstem}")

        return header_data, df, constants_df, modelstem

In [ ]:
if False:
    input_folder = OUTPUT_FOLDER
    output_folder = OUTPUT_FOLDER

    export_data_files = list(input_folder.rglob("*_exported_data.txt"))

    print("List of exported data files to be processed:")
    for txtfile in export_data_files:
        print(txtfile.stem)

    print("\nCurrently processing...\n")
    for txtfile in export_data_files:
        print(txtfile.stem)
        try:
            header_data, df, constants_df, modelstem = import_solution_data(
                filename = txtfile.name,
                input_folder = txtfile.parent,
            )
            constants_df.to_csv(output_folder / f"{modelstem}_constant_columns.csv", index=False)

            tl.log_message(f"Searching columns height, width and length in the imported data")
            try:
                height = df['root.epilayer_height (m)'].iloc[0]
                width = 0.5 * df['root.conductor_all_width (m)'].iloc[0]
                length = 0.5 * df['root.conductor_all_length (m)'].iloc[0]
            except KeyError as e:
                tl.log_message(f"ERROR appeared while searching for height, width and length in the imported data")
                height = 1e-6
                width = 0.5 * 500e-6
                length = 0.5* 500e-6
                tl.log_message(f"Using substitute values: height: {height}, width: {width}, length: {length}")

        # DEPTH INTERPOLATION
            print(f"Depth, ")
            interpolate_and_save(
                # path settings
                modelname = modelstem,
                output_folder = output_folder,
                interpolation_type = "depth",

                # grid settings
                grid_dict = {
                    'x': [0.0],
                    'y': [0.0],
                    },

                limit_dict = {
                    'z': [0, - height],
                    },

                # data settings
                df = df,
                x_axis_params = X_AXIS_PARAMS,
                y_axis_params = Y_AXIS_PARAMS,
                )

        # HOMOGENEITY INTERPOLATION
            print(f"Homogeneity, ")
            interpolate_and_save(
                # path settings
                modelname = modelstem,
                output_folder = output_folder,
                interpolation_type = "homogeneity",

                # grid settings
                grid_dict = {
                    'x': [0.0],
                    'z': np.linspace(0, -height, 11),
                    },

                limit_dict = {
                    'y': [1.2*width, -1.2*width],
                    },

                # data settings
                df = df,
                x_axis_params = X_AXIS_PARAMS,
                y_axis_params = Y_AXIS_PARAMS,
                )

        # LONGITUDINAL INTERPOLATION
            print(f"Longitudinal, ")
            interpolate_and_save(
                # path settings
                modelname = modelstem,
                output_folder = output_folder,
                interpolation_type = "longitudinal",

                # grid settings
                grid_dict = {
                    'y': [0.0],
                    'z': np.linspace(0, -height, 11),
                    },

                limit_dict = {
                    'x': [1.2*width, -1.2*width],
                    },

                # data settings
                df = df,
                x_axis_params = X_AXIS_PARAMS,
                y_axis_params = Y_AXIS_PARAMS,
                )

        # ERROR LOGGING
        except Exception as e:
            with open(output_folder / 'error_log.txt', 'a') as f:
                f.write(f"Error occurred while processing {txtfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {txtfile.name}")
        print(f"Finished.\n")

# END

In [ ]:
tl.log_message("Reached the end of the script.")